
# PanTiny — Local Full-Scene GeoTIFF Inference + Wald Test Export

هذه الـNotebook مخصصة للتشغيل محليًا من **VS Code / Jupyter** بعد انتهاء تدريب PanTiny.

تقوم بمهمتين مستقلتين:

1. **تصدير Test Data** لكل عينة Wald:
   - `input_lr_ms.tif`
   - `input_pan.tif`
   - `target_hr_ms.tif`
   - `output_pantiny_hr_ms.tif`
   - `baseline_bicubic_hr_ms.tif`

2. **تشغيل PanTiny على الصورة الأصلية كاملة**:
   - Input MS: `6 × 1536 × 1804`
   - Input PAN: `1 × 6144 × 7216`
   - Output HR-MS: `6 × 6144 × 7216`
   - يحفظ الناتج كـGeoTIFF مع CRS وTransform وBounds الخاصة بصورة PAN.

الناتج الكامل يتم بنظام **Overlapping Tiled Inference + Weighted Blending** لتقليل الفواصل بين الـTiles، مع استخدام ملفات `memmap` على القرص بدل استهلاك RAM كبيرة.



## 1. تثبيت المكتبات

شغّل هذه الخلية مرة واحدة داخل بيئة المشروع المحلية.  
لا تعيد تثبيت PyTorch إذا كان إصدار CUDA الصحيح مثبتًا بالفعل.


In [1]:

# شغّل مرة واحدة فقط داخل نفس Python environment المختارة في VS Code:
# %pip install rasterio numpy pandas matplotlib tqdm

import os
import gc
import json
import math
import shutil
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

import rasterio
from affine import Affine
from rasterio.enums import Resampling
from rasterio.windows import Window
from rasterio.windows import transform as window_transform
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

USE_AMP = torch.cuda.is_available()

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2,
        ),
        "GB",
    )
else:
    print(
        "Warning: CUDA غير متاحة. "
        "التشغيل الكامل على CPU سيكون بطيئًا جدًا."
    )


PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4
VRAM: 14.56 GB



## 2. إعداد مسارات المشروع

عدّل `PROJECT_OVERRIDE` فقط إذا لم تكن فاتح مجلد المشروع نفسه في VS Code.


In [2]:

# ------------------------------------------------------------
# عدّل هذا المسار فقط عند الحاجة
# ------------------------------------------------------------

PROJECT_OVERRIDE = Path(
    r"D:\Super_Resolution_28-07-2026"
)

# لو فتحت VS Code داخل مجلد المشروع ويمكنك استخدام Path.cwd():
# PROJECT_OVERRIDE = None

PROJECT = (
    PROJECT_OVERRIDE
    if PROJECT_OVERRIDE is not None
    else Path.cwd()
)

PROJECT = PROJECT.expanduser().resolve()

MS_PATH = (
    PROJECT
    / "Aligned_Data"
    / "MS_aligned.tif"
)

PAN_PATH = (
    PROJECT
    / "Aligned_Data"
    / "PAN_aligned.tif"
)

STATS_PATH = (
    PROJECT
    / "Fusion_Baseline_Results"
    / "train_normalization_stats.json"
)

CHECKPOINT_PATH = (
    PROJECT
    / "PanTiny_6Band_Results"
    / "best_pantiny_small_6band.pth"
)

TEST_DIR = (
    PROJECT
    / "Wald_Data_GSD"
    / "Patches"
    / "test"
)

OUTPUT_ROOT = (
    PROJECT
    / "PanTiny_TIFF_Inference_Results"
)

TEST_TIFF_DIR = (
    OUTPUT_ROOT
    / "Wald_Test_GeoTIFFs"
)

FULL_OUTPUT_DIR = (
    OUTPUT_ROOT
    / "Full_Scene"
)

SCRATCH_DIR = (
    OUTPUT_ROOT
    / "_scratch_memmap"
)

for directory in [
    OUTPUT_ROOT,
    TEST_TIFF_DIR,
    FULL_OUTPUT_DIR,
    SCRATCH_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

# ------------------------------------------------------------
# تشغيل الأقسام
# ------------------------------------------------------------

EXPORT_WALD_TEST_TIFS = True
RUN_FULL_SCENE = True

# "crop" لاختبار سريع، أو "full" للصورة كاملة.
RUN_MODE = "full"

# يستخدم فقط عندما RUN_MODE = "crop"
CROP_LR_SIZE = 256

SCALE = 4
MS_BANDS = 6

# قيم آمنة مبدئيًا لمعظم كروت 8 GB أو أكثر.
# لو حدث CUDA out of memory، غيّر TILE_LR إلى 128.
TILE_LR = 192
OVERLAP_LR = 32
STRIDE_LR = TILE_LR - OVERLAP_LR

SAVE_FLOAT32_TIF = True
SAVE_UINT16_TIF = True
SAVE_RGB_PREVIEW = True
DELETE_SCRATCH_AFTER_SUCCESS = True

# ترتيب RGB للعرض فقط.
# تأكد أنه يطابق ترتيب Bands في المستشعر المحلي.
RGB_ZERO_BASED = [2, 1, 0]

required = {
    "Project": PROJECT,
    "Aligned MS": MS_PATH,
    "Aligned PAN": PAN_PATH,
    "Normalization stats": STATS_PATH,
    "PanTiny checkpoint": CHECKPOINT_PATH,
}

if EXPORT_WALD_TEST_TIFS:
    required["Wald test directory"] = TEST_DIR

for name, path in required.items():
    print(
        f"{name:26s}",
        "→",
        path.exists(),
        "→",
        path,
    )

missing = [
    f"{name}: {path}"
    for name, path in required.items()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required project paths:\n"
        + "\n".join(missing)
    )

print("\nProject:", PROJECT)
print("Run mode:", RUN_MODE)
print("Tile LR / overlap / stride:", TILE_LR, OVERLAP_LR, STRIDE_LR)


Project                    → True → /content/D:\Super_Resolution_28-07-2026
Aligned MS                 → False → /content/D:\Super_Resolution_28-07-2026/Aligned_Data/MS_aligned.tif
Aligned PAN                → False → /content/D:\Super_Resolution_28-07-2026/Aligned_Data/PAN_aligned.tif
Normalization stats        → False → /content/D:\Super_Resolution_28-07-2026/Fusion_Baseline_Results/train_normalization_stats.json
PanTiny checkpoint         → False → /content/D:\Super_Resolution_28-07-2026/PanTiny_6Band_Results/best_pantiny_small_6band.pth
Wald test directory        → False → /content/D:\Super_Resolution_28-07-2026/Wald_Data_GSD/Patches/test


FileNotFoundError: Missing required project paths:
Aligned MS: /content/D:\Super_Resolution_28-07-2026/Aligned_Data/MS_aligned.tif
Aligned PAN: /content/D:\Super_Resolution_28-07-2026/Aligned_Data/PAN_aligned.tif
Normalization stats: /content/D:\Super_Resolution_28-07-2026/Fusion_Baseline_Results/train_normalization_stats.json
PanTiny checkpoint: /content/D:\Super_Resolution_28-07-2026/PanTiny_6Band_Results/best_pantiny_small_6band.pth
Wald test directory: /content/D:\Super_Resolution_28-07-2026/Wald_Data_GSD/Patches/test


## 3. Normalization المطابقة للتدريب


In [ ]:

with open(
    STATS_PATH,
    "r",
    encoding="utf-8",
) as file:
    normalization_stats = json.load(file)

MS_LOW = np.asarray(
    [
        item["p01"]
        for item in normalization_stats["ms"]
    ],
    dtype=np.float32,
)[:, None, None]

MS_HIGH = np.asarray(
    [
        item["p99"]
        for item in normalization_stats["ms"]
    ],
    dtype=np.float32,
)[:, None, None]

PAN_LOW = np.float32(
    normalization_stats["pan"]["p01"]
)

PAN_HIGH = np.float32(
    normalization_stats["pan"]["p99"]
)

if MS_LOW.shape != (MS_BANDS, 1, 1):
    raise ValueError(
        f"Expected {MS_BANDS} MS normalization bands, "
        f"found {MS_LOW.shape[0]}"
    )


def normalize_ms(
    array: np.ndarray,
) -> np.ndarray:
    return np.clip(
        (
            array.astype(np.float32)
            - MS_LOW
        )
        / np.maximum(
            MS_HIGH - MS_LOW,
            1e-6,
        ),
        0.0,
        1.0,
    )


def normalize_pan(
    array: np.ndarray,
) -> np.ndarray:
    return np.clip(
        (
            array.astype(np.float32)
            - PAN_LOW
        )
        / max(
            float(PAN_HIGH - PAN_LOW),
            1e-6,
        ),
        0.0,
        1.0,
    )


def denormalize_ms(
    normalized_chw: np.ndarray,
) -> np.ndarray:
    return (
        normalized_chw.astype(np.float32)
        * (MS_HIGH - MS_LOW)
        + MS_LOW
    )


print("MS p01:", MS_LOW[:, 0, 0])
print("MS p99:", MS_HIGH[:, 0, 0])
print("PAN p01 / p99:", float(PAN_LOW), float(PAN_HIGH))



## 4. معمارية PanTiny المطابقة للـCheckpoint


In [ ]:

class LayerNorm2d(nn.Module):
    def __init__(
        self,
        channels: int,
    ):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(channels)
        )

        self.bias = nn.Parameter(
            torch.zeros(channels)
        )

    def forward(
        self,
        tensor: torch.Tensor,
    ) -> torch.Tensor:
        mean = tensor.mean(
            dim=1,
            keepdim=True,
        )

        variance = tensor.var(
            dim=1,
            keepdim=True,
            unbiased=False,
        )

        normalized = (
            tensor - mean
        ) / torch.sqrt(
            variance + 1e-5
        )

        return (
            normalized
            * self.weight[
                None,
                :,
                None,
                None,
            ]
            + self.bias[
                None,
                :,
                None,
                None,
            ]
        )


class ChannelAttention(nn.Module):
    def __init__(
        self,
        dim: int,
        heads: int,
    ):
        super().__init__()

        if dim % heads != 0:
            raise ValueError(
                "dim must be divisible by heads"
            )

        self.dim = dim
        self.heads = heads

        self.temperature = nn.Parameter(
            torch.ones(
                heads,
                1,
                1,
            )
        )

        self.qkv = nn.Conv2d(
            dim,
            dim * 3,
            kernel_size=1,
            bias=False,
        )

        self.qkv_dwconv = nn.Conv2d(
            dim * 3,
            dim * 3,
            kernel_size=3,
            stride=1,
            padding=1,
            groups=dim * 3,
            bias=False,
        )

        self.project_out = nn.Conv2d(
            dim,
            dim,
            kernel_size=1,
            bias=False,
        )

    def forward(
        self,
        tensor: torch.Tensor,
    ) -> torch.Tensor:
        batch, channels, height, width = tensor.shape

        qkv = self.qkv_dwconv(
            self.qkv(tensor)
        )

        query, key, value = qkv.chunk(
            3,
            dim=1,
        )

        channels_per_head = (
            channels // self.heads
        )

        query = query.reshape(
            batch,
            self.heads,
            channels_per_head,
            height * width,
        )

        key = key.reshape(
            batch,
            self.heads,
            channels_per_head,
            height * width,
        )

        value = value.reshape(
            batch,
            self.heads,
            channels_per_head,
            height * width,
        )

        query = F.normalize(
            query,
            dim=-1,
        )

        key = F.normalize(
            key,
            dim=-1,
        )

        attention = (
            query
            @ key.transpose(-2, -1)
        ) * self.temperature

        attention = attention.softmax(
            dim=-1
        )

        output = attention @ value

        output = output.reshape(
            batch,
            channels,
            height,
            width,
        )

        return self.project_out(
            output
        )


class GatedDConvFeedForward(nn.Module):
    def __init__(
        self,
        dim: int,
        expansion: float,
    ):
        super().__init__()

        hidden = int(
            dim * expansion
        )

        self.project_in = nn.Conv2d(
            dim,
            hidden * 2,
            kernel_size=1,
            bias=False,
        )

        self.depthwise = nn.Conv2d(
            hidden * 2,
            hidden * 2,
            kernel_size=3,
            stride=1,
            padding=1,
            groups=hidden * 2,
            bias=False,
        )

        self.project_out = nn.Conv2d(
            hidden,
            dim,
            kernel_size=1,
            bias=False,
        )

    def forward(
        self,
        tensor: torch.Tensor,
    ) -> torch.Tensor:
        tensor = self.depthwise(
            self.project_in(tensor)
        )

        first, second = tensor.chunk(
            2,
            dim=1,
        )

        tensor = (
            F.gelu(first)
            * second
        )

        return self.project_out(
            tensor
        )


class TransformerBlock(nn.Module):
    def __init__(
        self,
        dim: int,
        heads: int,
        expansion: float,
    ):
        super().__init__()

        self.norm1 = LayerNorm2d(dim)

        self.attention = ChannelAttention(
            dim,
            heads,
        )

        self.norm2 = LayerNorm2d(dim)

        self.ffn = GatedDConvFeedForward(
            dim,
            expansion,
        )

    def forward(
        self,
        tensor: torch.Tensor,
    ) -> torch.Tensor:
        tensor = (
            tensor
            + self.attention(
                self.norm1(tensor)
            )
        )

        tensor = (
            tensor
            + self.ffn(
                self.norm2(tensor)
            )
        )

        return tensor


class EnhancedConv(nn.Module):
    def __init__(
        self,
        dim: int,
    ):
        super().__init__()

        self.body = nn.Sequential(
            nn.Conv2d(
                dim,
                dim,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.GELU(),
            nn.Conv2d(
                dim,
                dim,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
        )

    def forward(
        self,
        tensor: torch.Tensor,
    ) -> torch.Tensor:
        return (
            tensor
            + self.body(tensor)
        )


class PanTiny6Band(nn.Module):
    def __init__(
        self,
        ms_bands: int,
        dim: int,
        depth: int,
        heads: int,
        ffn_expansion: float,
    ):
        super().__init__()

        self.ms_encoder = nn.Sequential(
            nn.Conv2d(
                ms_bands,
                dim,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.GELU(),
        )

        self.pan_projection = nn.Conv2d(
            1,
            dim,
            kernel_size=3,
            padding=1,
            bias=False,
        )

        self.fusion = nn.Sequential(
            nn.Conv2d(
                dim * 2,
                dim,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.GELU(),
        )

        self.body = nn.Sequential(
            *[
                TransformerBlock(
                    dim=dim,
                    heads=heads,
                    expansion=ffn_expansion,
                )
                for _ in range(depth)
            ]
        )

        self.enhanced_conv = EnhancedConv(
            dim
        )

        self.output = nn.Conv2d(
            dim,
            ms_bands,
            kernel_size=3,
            padding=1,
            bias=True,
        )

    def forward(
        self,
        lr_ms: torch.Tensor,
        pan: torch.Tensor,
    ) -> torch.Tensor:
        ms_bicubic = F.interpolate(
            lr_ms,
            size=pan.shape[-2:],
            mode="bicubic",
            align_corners=False,
        )

        ms_features = self.ms_encoder(
            ms_bicubic
        )

        pan_features = self.pan_projection(
            pan
        )

        fused = self.fusion(
            torch.cat(
                [
                    ms_features,
                    pan_features,
                ],
                dim=1,
            )
        )

        body_features = (
            fused
            + self.body(fused)
        )

        refined = self.enhanced_conv(
            body_features
        )

        residual = self.output(
            refined
        )

        return (
            ms_bicubic
            + residual
        ).clamp(
            0.0,
            1.0,
        )



## 5. تحميل أفضل Checkpoint


In [ ]:

def load_torch_checkpoint(
    path: Path,
):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE,
        )


checkpoint = load_torch_checkpoint(
    CHECKPOINT_PATH
)

model_config = checkpoint.get(
    "model_config",
    {
        "dim": 24,
        "depth": 4,
        "heads": 3,
        "ffn_expansion": 2.0,
    },
)

model = PanTiny6Band(
    ms_bands=MS_BANDS,
    dim=int(
        model_config["dim"]
    ),
    depth=int(
        model_config["depth"]
    ),
    heads=int(
        model_config["heads"]
    ),
    ffn_expansion=float(
        model_config["ffn_expansion"]
    ),
).to(DEVICE)

state_dict = checkpoint.get(
    "model_state_dict",
    checkpoint.get(
        "ema_state_dict",
        checkpoint,
    ),
)

clean_state_dict = {}

for key, value in state_dict.items():
    clean_key = (
        key[7:]
        if key.startswith("module.")
        else key
    )

    clean_state_dict[
        clean_key
    ] = value

model.load_state_dict(
    clean_state_dict,
    strict=True,
)

model.eval()

PARAMETER_COUNT = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Checkpoint:", CHECKPOINT_PATH)
print("Checkpoint epoch:", checkpoint.get("epoch", "unknown"))
print("Model config:", model_config)
print("Parameters:", f"{PARAMETER_COUNT:,}")

# Forward smoke test
with torch.inference_mode():
    smoke_lr = torch.zeros(
        1,
        MS_BANDS,
        32,
        32,
        device=DEVICE,
    )

    smoke_pan = torch.zeros(
        1,
        1,
        32 * SCALE,
        32 * SCALE,
        device=DEVICE,
    )

    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=USE_AMP,
    ):
        smoke_output = model(
            smoke_lr,
            smoke_pan,
        )

print("Smoke output:", tuple(smoke_output.shape))

del smoke_lr
del smoke_pan
del smoke_output

if torch.cuda.is_available():
    torch.cuda.empty_cache()



# القسم الأول — تصدير Wald Test Input / Target / Output إلى GeoTIFF

كل عينة تحفظ داخل مجلد مستقل وبنفس الإسناد الجغرافي المستخرج من `x` و`y` الموجودين داخل ملف NPZ.


In [ ]:

def make_gtiff_profile(
    reference_profile: dict,
    *,
    width: int,
    height: int,
    count: int,
    dtype: str,
    transform,
    crs,
    nodata=0,
):
    profile = reference_profile.copy()

    for key in [
        "blockxsize",
        "blockysize",
        "photometric",
        "interleave",
    ]:
        profile.pop(
            key,
            None,
        )

    predictor = (
        3
        if np.dtype(dtype).kind == "f"
        else 2
    )

    profile.update(
        driver="GTiff",
        width=width,
        height=height,
        count=count,
        dtype=dtype,
        transform=transform,
        crs=crs,
        nodata=nodata,
        compress="deflate",
        predictor=predictor,
        tiled=True,
        blockxsize=256,
        blockysize=256,
        interleave="band",
        BIGTIFF="IF_SAFER",
    )

    return profile


def write_array_geotiff(
    output_path: Path,
    array_chw: np.ndarray,
    *,
    reference_profile: dict,
    transform,
    crs,
    descriptions=None,
    tags=None,
):
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    array_chw = np.asarray(
        array_chw
    )

    if array_chw.ndim == 2:
        array_chw = array_chw[
            None
        ]

    count, height, width = (
        array_chw.shape
    )

    profile = make_gtiff_profile(
        reference_profile,
        width=width,
        height=height,
        count=count,
        dtype=str(
            array_chw.dtype
        ),
        transform=transform,
        crs=crs,
        nodata=0,
    )

    with rasterio.open(
        output_path,
        "w",
        **profile,
    ) as destination:
        destination.write(
            array_chw
        )

        if descriptions is not None:
            for band_index, description in enumerate(
                descriptions,
                start=1,
            ):
                destination.set_band_description(
                    band_index,
                    str(description),
                )

        if tags:
            destination.update_tags(
                **tags
            )


def get_npz_xy(
    sample,
    path: Path,
):
    if (
        "x" in sample.files
        and "y" in sample.files
    ):
        return (
            int(sample["x"]),
            int(sample["y"]),
        )

    import re

    match = re.search(
        r"_y(\d+)_x(\d+)",
        path.stem,
    )

    if match is None:
        raise ValueError(
            f"Cannot determine x/y for {path.name}"
        )

    y = int(
        match.group(1)
    )

    x = int(
        match.group(2)
    )

    return x, y


test_manifest_rows = []

if EXPORT_WALD_TEST_TIFS:
    test_files = sorted(
        TEST_DIR.glob(
            "*.npz"
        )
    )

    if not test_files:
        raise RuntimeError(
            f"No test NPZ files found in {TEST_DIR}"
        )

    with rasterio.open(
        MS_PATH
    ) as ms_source:
        ms_profile = (
            ms_source.profile.copy()
        )

        ms_transform = (
            ms_source.transform
        )

        ms_crs = ms_source.crs

        ms_descriptions = [
            description
            if description
            else f"MS_Band_{index}"
            for index, description in enumerate(
                ms_source.descriptions,
                start=1,
            )
        ]

    for test_path in tqdm(
        test_files,
        desc="Exporting Wald test GeoTIFFs",
    ):
        with np.load(
            test_path
        ) as sample:
            lr_ms_dn = sample[
                "lr_ms"
            ].copy()

            pan_dn = sample[
                "pan"
            ].copy()

            target_dn = sample[
                "target_ms"
            ].copy()

            x, y = get_npz_xy(
                sample,
                test_path,
            )

        if pan_dn.ndim == 2:
            pan_dn = pan_dn[
                None
            ]

        lr_norm = torch.from_numpy(
            normalize_ms(
                lr_ms_dn
            )
        )[
            None
        ].float().to(
            DEVICE
        )

        pan_norm = torch.from_numpy(
            normalize_pan(
                pan_dn
            )
        )[
            None
        ].float().to(
            DEVICE
        )

        with torch.inference_mode():
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                prediction_norm = model(
                    lr_norm,
                    pan_norm,
                )

                bicubic_norm = F.interpolate(
                    lr_norm,
                    size=pan_norm.shape[-2:],
                    mode="bicubic",
                    align_corners=False,
                ).clamp(
                    0.0,
                    1.0,
                )

        prediction_dn = denormalize_ms(
            prediction_norm[
                0
            ].float().cpu().numpy()
        )

        bicubic_dn = denormalize_ms(
            bicubic_norm[
                0
            ].float().cpu().numpy()
        )

        prediction_u16 = np.clip(
            np.rint(
                prediction_dn
            ),
            0,
            65535,
        ).astype(
            np.uint16
        )

        bicubic_u16 = np.clip(
            np.rint(
                bicubic_dn
            ),
            0,
            65535,
        ).astype(
            np.uint16
        )

        lr_ms_u16 = np.clip(
            np.rint(
                lr_ms_dn
            ),
            0,
            65535,
        ).astype(
            np.uint16
        )

        pan_u16 = np.clip(
            np.rint(
                pan_dn
            ),
            0,
            65535,
        ).astype(
            np.uint16
        )

        target_u16 = np.clip(
            np.rint(
                target_dn
            ),
            0,
            65535,
        ).astype(
            np.uint16
        )

        target_height = int(
            target_u16.shape[-2]
        )

        target_width = int(
            target_u16.shape[-1]
        )

        target_transform = window_transform(
            Window(
                x,
                y,
                target_width,
                target_height,
            ),
            ms_transform,
        )

        lr_transform = (
            target_transform
            * Affine.scale(
                SCALE,
                SCALE,
            )
        )

        sample_dir = (
            TEST_TIFF_DIR
            / test_path.stem
        )

        tags = {
            "MODEL": "PanTiny 6-Band",
            "CHECKPOINT": str(
                CHECKPOINT_PATH
            ),
            "SOURCE_NPZ": str(
                test_path
            ),
            "SCALE_FACTOR": str(
                SCALE
            ),
        }

        write_array_geotiff(
            sample_dir
            / "input_lr_ms.tif",
            lr_ms_u16,
            reference_profile=ms_profile,
            transform=lr_transform,
            crs=ms_crs,
            descriptions=ms_descriptions,
            tags={
                **tags,
                "ROLE": "Wald LR-MS input",
            },
        )

        write_array_geotiff(
            sample_dir
            / "input_pan.tif",
            pan_u16,
            reference_profile=ms_profile,
            transform=target_transform,
            crs=ms_crs,
            descriptions=[
                "Reduced-resolution PAN"
            ],
            tags={
                **tags,
                "ROLE": "Wald PAN input",
            },
        )

        write_array_geotiff(
            sample_dir
            / "target_hr_ms.tif",
            target_u16,
            reference_profile=ms_profile,
            transform=target_transform,
            crs=ms_crs,
            descriptions=ms_descriptions,
            tags={
                **tags,
                "ROLE": "Wald HR-MS target",
            },
        )

        write_array_geotiff(
            sample_dir
            / "output_pantiny_hr_ms.tif",
            prediction_u16,
            reference_profile=ms_profile,
            transform=target_transform,
            crs=ms_crs,
            descriptions=ms_descriptions,
            tags={
                **tags,
                "ROLE": "PanTiny HR-MS prediction",
            },
        )

        write_array_geotiff(
            sample_dir
            / "baseline_bicubic_hr_ms.tif",
            bicubic_u16,
            reference_profile=ms_profile,
            transform=target_transform,
            crs=ms_crs,
            descriptions=ms_descriptions,
            tags={
                **tags,
                "ROLE": "Bicubic HR-MS baseline",
            },
        )

        test_manifest_rows.append(
            {
                "sample": test_path.stem,
                "x": x,
                "y": y,
                "lr_ms": str(
                    sample_dir
                    / "input_lr_ms.tif"
                ),
                "pan": str(
                    sample_dir
                    / "input_pan.tif"
                ),
                "target": str(
                    sample_dir
                    / "target_hr_ms.tif"
                ),
                "prediction": str(
                    sample_dir
                    / "output_pantiny_hr_ms.tif"
                ),
                "bicubic": str(
                    sample_dir
                    / "baseline_bicubic_hr_ms.tif"
                ),
            }
        )

        del lr_norm
        del pan_norm
        del prediction_norm
        del bicubic_norm

    test_manifest = pd.DataFrame(
        test_manifest_rows
    )

    test_manifest_path = (
        TEST_TIFF_DIR
        / "wald_test_tiff_manifest.csv"
    )

    test_manifest.to_csv(
        test_manifest_path,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        "Exported test samples:",
        len(test_manifest)
    )

    print(
        "Test manifest:",
        test_manifest_path
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

else:
    print(
        "Wald test export skipped."
    )



# القسم الثاني — الصورة الأصلية كاملة إلى PanTiny GeoTIFF كامل

لا يوجد Target حقيقي للصورة الكاملة عند دقة PAN؛ لذلك هذا القسم يحفظ الناتج العملي ولا يحسب PSNR للصورة الكاملة.


In [ ]:

def validate_full_scene_pair(
    ms_path: Path,
    pan_path: Path,
    scale: int,
):
    with rasterio.open(
        ms_path
    ) as ms_source, rasterio.open(
        pan_path
    ) as pan_source:
        if ms_source.count != MS_BANDS:
            raise ValueError(
                f"MS must contain {MS_BANDS} bands, "
                f"found {ms_source.count}"
            )

        if pan_source.count != 1:
            raise ValueError(
                f"PAN must contain one band, "
                f"found {pan_source.count}"
            )

        if (
            pan_source.width
            != ms_source.width * scale
            or pan_source.height
            != ms_source.height * scale
        ):
            raise ValueError(
                "PAN dimensions are not exactly "
                f"{scale}× MS dimensions.\n"
                f"MS: {ms_source.width}×{ms_source.height}\n"
                f"PAN: {pan_source.width}×{pan_source.height}"
            )

        if ms_source.crs != pan_source.crs:
            raise ValueError(
                "MS and PAN CRS values are different."
            )

        bounds_difference = np.max(
            np.abs(
                np.asarray(
                    ms_source.bounds
                )
                - np.asarray(
                    pan_source.bounds
                )
            )
        )

        return {
            "ms_shape": (
                ms_source.count,
                ms_source.height,
                ms_source.width,
            ),
            "pan_shape": (
                pan_source.count,
                pan_source.height,
                pan_source.width,
            ),
            "ms_dtype": ms_source.dtypes,
            "pan_dtype": pan_source.dtypes,
            "crs": str(ms_source.crs),
            "ms_transform": tuple(
                ms_source.transform
            ),
            "pan_transform": tuple(
                pan_source.transform
            ),
            "ms_bounds": tuple(
                ms_source.bounds
            ),
            "pan_bounds": tuple(
                pan_source.bounds
            ),
            "bounds_max_abs_difference": float(
                bounds_difference
            ),
        }


full_scene_metadata = validate_full_scene_pair(
    MS_PATH,
    PAN_PATH,
    SCALE,
)

print(
    json.dumps(
        full_scene_metadata,
        indent=2,
    )
)

with rasterio.open(
    MS_PATH
) as ms_source:
    full_lr_width = (
        ms_source.width
    )

    full_lr_height = (
        ms_source.height
    )

if RUN_MODE == "full":
    lr_x0 = 0
    lr_y0 = 0
    lr_width = full_lr_width
    lr_height = full_lr_height

elif RUN_MODE == "crop":
    lr_width = min(
        CROP_LR_SIZE,
        full_lr_width,
    )

    lr_height = min(
        CROP_LR_SIZE,
        full_lr_height,
    )

    lr_x0 = (
        full_lr_width - lr_width
    ) // 2

    lr_y0 = (
        full_lr_height - lr_height
    ) // 2

else:
    raise ValueError(
        'RUN_MODE must be "crop" or "full".'
    )

hr_width = lr_width * SCALE
hr_height = lr_height * SCALE

print(
    "\nSelected LR region:",
    {
        "x": lr_x0,
        "y": lr_y0,
        "width": lr_width,
        "height": lr_height,
    },
)

print(
    "Expected HR output:",
    (
        MS_BANDS,
        hr_height,
        hr_width,
    ),
)


In [ ]:

def tile_positions(
    length: int,
    tile: int,
    stride: int,
):
    if length <= tile:
        return [0]

    values = list(
        range(
            0,
            length - tile + 1,
            stride,
        )
    )

    final_position = (
        length - tile
    )

    if values[-1] != final_position:
        values.append(
            final_position
        )

    return values


def blend_weight(
    height: int,
    width: int,
):
    y_weight = np.hanning(
        height
    )

    x_weight = np.hanning(
        width
    )

    weight = np.outer(
        y_weight,
        x_weight,
    ).astype(
        np.float32
    )

    return np.maximum(
        weight,
        1e-3,
    )


tile_lr_width = min(
    TILE_LR,
    lr_width,
)

tile_lr_height = min(
    TILE_LR,
    lr_height,
)

tile_hr_width = (
    tile_lr_width
    * SCALE
)

tile_hr_height = (
    tile_lr_height
    * SCALE
)

x_positions = tile_positions(
    lr_width,
    tile_lr_width,
    min(
        STRIDE_LR,
        tile_lr_width,
    ),
)

y_positions = tile_positions(
    lr_height,
    tile_lr_height,
    min(
        STRIDE_LR,
        tile_lr_height,
    ),
)

tile_weight = blend_weight(
    tile_hr_height,
    tile_hr_width,
)

total_tiles = (
    len(x_positions)
    * len(y_positions)
)

print("LR tile:", tile_lr_width, tile_lr_height)
print("HR tile:", tile_hr_width, tile_hr_height)
print("Tiles:", total_tiles)


In [ ]:

def run_tiled_full_scene_inference():
    if not RUN_FULL_SCENE:
        print(
            "Full-scene inference skipped."
        )

        return None

    if SCRATCH_DIR.exists():
        shutil.rmtree(
            SCRATCH_DIR
        )

    SCRATCH_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    prediction_sum_path = (
        SCRATCH_DIR
        / "prediction_sum.float32.dat"
    )

    weight_sum_path = (
        SCRATCH_DIR
        / "weight_sum.float32.dat"
    )

    prediction_sum = np.memmap(
        prediction_sum_path,
        dtype=np.float32,
        mode="w+",
        shape=(
            MS_BANDS,
            hr_height,
            hr_width,
        ),
    )

    weight_sum = np.memmap(
        weight_sum_path,
        dtype=np.float32,
        mode="w+",
        shape=(
            hr_height,
            hr_width,
        ),
    )

    prediction_sum[:] = 0.0
    weight_sum[:] = 0.0

    start_time = time.time()

    progress = tqdm(
        total=total_tiles,
        desc="PanTiny full-scene tiles",
    )

    with rasterio.open(
        MS_PATH
    ) as ms_source, rasterio.open(
        PAN_PATH
    ) as pan_source:
        for local_y in y_positions:
            for local_x in x_positions:
                global_lr_x = (
                    lr_x0
                    + local_x
                )

                global_lr_y = (
                    lr_y0
                    + local_y
                )

                ms_tile = ms_source.read(
                    window=Window(
                        global_lr_x,
                        global_lr_y,
                        tile_lr_width,
                        tile_lr_height,
                    )
                )

                pan_tile = pan_source.read(
                    1,
                    window=Window(
                        global_lr_x
                        * SCALE,
                        global_lr_y
                        * SCALE,
                        tile_hr_width,
                        tile_hr_height,
                    ),
                )[
                    None
                ]

                ms_tensor = torch.from_numpy(
                    normalize_ms(
                        ms_tile
                    )
                )[
                    None
                ].float().to(
                    DEVICE,
                    non_blocking=True,
                )

                pan_tensor = torch.from_numpy(
                    normalize_pan(
                        pan_tile
                    )
                )[
                    None
                ].float().to(
                    DEVICE,
                    non_blocking=True,
                )

                with torch.inference_mode():
                    with torch.autocast(
                        device_type=DEVICE.type,
                        dtype=torch.float16,
                        enabled=USE_AMP,
                    ):
                        prediction = model(
                            ms_tensor,
                            pan_tensor,
                        )

                prediction = (
                    prediction[
                        0
                    ]
                    .float()
                    .cpu()
                    .numpy()
                )

                if not np.isfinite(
                    prediction
                ).all():
                    raise FloatingPointError(
                        "NaN or Inf detected at "
                        f"LR tile x={global_lr_x}, "
                        f"y={global_lr_y}"
                    )

                output_x = (
                    local_x
                    * SCALE
                )

                output_y = (
                    local_y
                    * SCALE
                )

                prediction_sum[
                    :,
                    output_y:
                    output_y + tile_hr_height,
                    output_x:
                    output_x + tile_hr_width,
                ] += (
                    prediction
                    * tile_weight[
                        None
                    ]
                )

                weight_sum[
                    output_y:
                    output_y + tile_hr_height,
                    output_x:
                    output_x + tile_hr_width,
                ] += tile_weight

                del ms_tensor
                del pan_tensor
                del prediction

                progress.update(1)

    progress.close()

    prediction_sum.flush()
    weight_sum.flush()

    elapsed_seconds = (
        time.time()
        - start_time
    )

    print(
        "Tiled inference completed in:",
        round(
            elapsed_seconds / 60,
            2,
        ),
        "minutes",
    )

    print(
        "Minimum accumulated weight:",
        float(
            np.min(
                weight_sum
            )
        ),
    )

    return {
        "prediction_sum": prediction_sum,
        "weight_sum": weight_sum,
        "prediction_sum_path": prediction_sum_path,
        "weight_sum_path": weight_sum_path,
        "elapsed_seconds": elapsed_seconds,
    }


full_inference = (
    run_tiled_full_scene_inference()
)


In [ ]:

def output_transform_and_profile():
    with rasterio.open(
        PAN_PATH
    ) as pan_source:
        pan_profile = (
            pan_source.profile.copy()
        )

        pan_crs = pan_source.crs

        if RUN_MODE == "full":
            transform = (
                pan_source.transform
            )

        else:
            transform = window_transform(
                Window(
                    lr_x0 * SCALE,
                    lr_y0 * SCALE,
                    hr_width,
                    hr_height,
                ),
                pan_source.transform,
            )

    return (
        transform,
        pan_profile,
        pan_crs,
    )


def create_full_output_profile(
    pan_profile: dict,
    *,
    dtype: str,
    transform,
    crs,
):
    profile = pan_profile.copy()

    for key in [
        "blockxsize",
        "blockysize",
        "photometric",
        "interleave",
    ]:
        profile.pop(
            key,
            None,
        )

    predictor = (
        3
        if np.dtype(dtype).kind == "f"
        else 2
    )

    profile.update(
        driver="GTiff",
        width=hr_width,
        height=hr_height,
        count=MS_BANDS,
        dtype=dtype,
        transform=transform,
        crs=crs,
        nodata=0,
        compress="deflate",
        predictor=predictor,
        tiled=True,
        blockxsize=256,
        blockysize=256,
        interleave="band",
        BIGTIFF="YES",
    )

    return profile


def copy_band_metadata(
    destination,
):
    with rasterio.open(
        MS_PATH
    ) as ms_source:
        for band_index in range(
            1,
            MS_BANDS + 1,
        ):
            description = (
                ms_source.descriptions[
                    band_index - 1
                ]
                or f"MS_Band_{band_index}"
            )

            destination.set_band_description(
                band_index,
                description,
            )

            source_tags = ms_source.tags(
                band_index
            )

            if source_tags:
                destination.update_tags(
                    band_index,
                    **source_tags,
                )


def write_full_scene_outputs(
    inference_result,
):
    if inference_result is None:
        return {}

    prediction_sum = inference_result[
        "prediction_sum"
    ]

    weight_sum = inference_result[
        "weight_sum"
    ]

    transform, pan_profile, crs = (
        output_transform_and_profile()
    )

    suffix = (
        "full"
        if RUN_MODE == "full"
        else "crop"
    )

    float_output = (
        FULL_OUTPUT_DIR
        / f"PanTiny_HRMS_6band_{suffix}_float32.tif"
    )

    uint16_output = (
        FULL_OUTPUT_DIR
        / f"PanTiny_HRMS_6band_{suffix}_uint16.tif"
    )

    float_destination = None
    uint16_destination = None

    try:
        if SAVE_FLOAT32_TIF:
            float_profile = (
                create_full_output_profile(
                    pan_profile,
                    dtype="float32",
                    transform=transform,
                    crs=crs,
                )
            )

            float_destination = rasterio.open(
                float_output,
                "w",
                **float_profile,
            )

            copy_band_metadata(
                float_destination
            )

        if SAVE_UINT16_TIF:
            uint16_profile = (
                create_full_output_profile(
                    pan_profile,
                    dtype="uint16",
                    transform=transform,
                    crs=crs,
                )
            )

            uint16_destination = rasterio.open(
                uint16_output,
                "w",
                **uint16_profile,
            )

            copy_band_metadata(
                uint16_destination
            )

        block_size = 512

        with rasterio.open(
            PAN_PATH
        ) as pan_source:
            for output_y in tqdm(
                range(
                    0,
                    hr_height,
                    block_size,
                ),
                desc="Writing full GeoTIFF",
            ):
                block_height = min(
                    block_size,
                    hr_height - output_y,
                )

                for output_x in range(
                    0,
                    hr_width,
                    block_size,
                ):
                    block_width = min(
                        block_size,
                        hr_width - output_x,
                    )

                    block_window = Window(
                        output_x,
                        output_y,
                        block_width,
                        block_height,
                    )

                    numerator = np.asarray(
                        prediction_sum[
                            :,
                            output_y:
                            output_y + block_height,
                            output_x:
                            output_x + block_width,
                        ],
                        dtype=np.float32,
                    )

                    denominator = np.asarray(
                        weight_sum[
                            output_y:
                            output_y + block_height,
                            output_x:
                            output_x + block_width,
                        ],
                        dtype=np.float32,
                    )

                    normalized_block = np.clip(
                        numerator
                        / np.maximum(
                            denominator[
                                None
                            ],
                            1e-8,
                        ),
                        0.0,
                        1.0,
                    )

                    dn_block = denormalize_ms(
                        normalized_block
                    )

                    pan_global_window = Window(
                        lr_x0
                        * SCALE
                        + output_x,
                        lr_y0
                        * SCALE
                        + output_y,
                        block_width,
                        block_height,
                    )

                    valid_mask = (
                        pan_source.read_masks(
                            1,
                            window=pan_global_window,
                        )
                        > 0
                    )

                    dn_block[
                        :,
                        ~valid_mask,
                    ] = 0.0

                    if float_destination is not None:
                        float_destination.write(
                            dn_block.astype(
                                np.float32
                            ),
                            window=block_window,
                        )

                    if uint16_destination is not None:
                        uint16_block = np.clip(
                            np.rint(
                                dn_block
                            ),
                            0,
                            65535,
                        ).astype(
                            np.uint16
                        )

                        uint16_destination.write(
                            uint16_block,
                            window=block_window,
                        )

        common_tags = {
            "MODEL": "PanTiny 6-Band",
            "CHECKPOINT": str(
                CHECKPOINT_PATH
            ),
            "CHECKPOINT_EPOCH": str(
                checkpoint.get(
                    "epoch",
                    "unknown",
                )
            ),
            "MODEL_PARAMETERS": str(
                PARAMETER_COUNT
            ),
            "SCALE_FACTOR": str(
                SCALE
            ),
            "RUN_MODE": RUN_MODE,
            "TILE_LR": str(
                TILE_LR
            ),
            "OVERLAP_LR": str(
                OVERLAP_LR
            ),
            "INPUT_MS": str(
                MS_PATH
            ),
            "INPUT_PAN": str(
                PAN_PATH
            ),
            "NORMALIZATION": "training p01-p99 per MS band and PAN",
            "GROUND_TRUTH_AVAILABLE": "No for full-resolution scene",
        }

        for destination in [
            float_destination,
            uint16_destination,
        ]:
            if destination is None:
                continue

            destination.update_tags(
                **common_tags
            )

            overview_factors = [
                factor
                for factor in [
                    2,
                    4,
                    8,
                    16,
                ]
                if (
                    hr_width // factor >= 1
                    and hr_height // factor >= 1
                )
            ]

            if overview_factors:
                destination.build_overviews(
                    overview_factors,
                    Resampling.average,
                )

                destination.update_tags(
                    ns="rio_overview",
                    resampling="average",
                )

    finally:
        if float_destination is not None:
            float_destination.close()

        if uint16_destination is not None:
            uint16_destination.close()

    outputs = {}

    if SAVE_FLOAT32_TIF:
        outputs[
            "float32_tif"
        ] = str(
            float_output
        )

    if SAVE_UINT16_TIF:
        outputs[
            "uint16_tif"
        ] = str(
            uint16_output
        )

    return outputs


full_output_paths = write_full_scene_outputs(
    full_inference
)

print(
    json.dumps(
        full_output_paths,
        indent=2,
    )
)


In [ ]:

def stretch_display_band(
    band: np.ndarray,
):
    band = band.astype(
        np.float32
    )

    finite = band[
        np.isfinite(
            band
        )
        & (band > 0)
    ]

    if finite.size == 0:
        return np.zeros_like(
            band,
            dtype=np.float32,
        )

    low, high = np.percentile(
        finite,
        [
            2,
            98,
        ],
    )

    return np.clip(
        (
            band - low
        )
        / max(
            float(
                high - low
            ),
            1e-6,
        ),
        0.0,
        1.0,
    )


preview_path = None

if (
    RUN_FULL_SCENE
    and SAVE_RGB_PREVIEW
    and full_output_paths
):
    preview_source_path = Path(
        full_output_paths.get(
            "uint16_tif",
            full_output_paths.get(
                "float32_tif"
            ),
        )
    )

    with rasterio.open(
        preview_source_path
    ) as source:
        preview_height = min(
            1200,
            source.height,
        )

        preview_width = max(
            1,
            round(
                source.width
                * preview_height
                / source.height
            ),
        )

        rgb_bands_one_based = [
            band_index + 1
            for band_index in RGB_ZERO_BASED
        ]

        preview_chw = source.read(
            rgb_bands_one_based,
            out_shape=(
                3,
                preview_height,
                preview_width,
            ),
            resampling=Resampling.bilinear,
        )

    preview_rgb = np.stack(
        [
            stretch_display_band(
                preview_chw[
                    channel_index
                ]
            )
            for channel_index in range(3)
        ],
        axis=-1,
    )

    preview_path = (
        FULL_OUTPUT_DIR
        / f"PanTiny_HRMS_6band_{RUN_MODE}_RGB_preview.png"
    )

    plt.figure(
        figsize=(
            15,
            11,
        )
    )

    plt.imshow(
        preview_rgb
    )

    plt.title(
        f"PanTiny Full-Scene Output — {RUN_MODE}"
    )

    plt.axis(
        "off"
    )

    plt.tight_layout()

    plt.savefig(
        preview_path,
        dpi=220,
        bbox_inches="tight",
    )

    plt.show()
    plt.close()

    print(
        "Preview:",
        preview_path
    )


In [ ]:

run_manifest = {
    "project": str(
        PROJECT
    ),
    "device": str(
        DEVICE
    ),
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "checkpoint": str(
        CHECKPOINT_PATH
    ),
    "checkpoint_epoch": checkpoint.get(
        "epoch",
        None,
    ),
    "model_config": model_config,
    "parameter_count": int(
        PARAMETER_COUNT
    ),
    "normalization_stats": str(
        STATS_PATH
    ),
    "input_ms": str(
        MS_PATH
    ),
    "input_pan": str(
        PAN_PATH
    ),
    "full_scene_metadata": full_scene_metadata,
    "run_mode": RUN_MODE,
    "selected_lr_region": {
        "x": int(
            lr_x0
        ),
        "y": int(
            lr_y0
        ),
        "width": int(
            lr_width
        ),
        "height": int(
            lr_height
        ),
    },
    "expected_output_shape": [
        MS_BANDS,
        int(
            hr_height
        ),
        int(
            hr_width
        ),
    ],
    "tile_lr": int(
        TILE_LR
    ),
    "overlap_lr": int(
        OVERLAP_LR
    ),
    "stride_lr": int(
        STRIDE_LR
    ),
    "test_geotiff_samples": len(
        test_manifest_rows
    ),
    "full_outputs": full_output_paths,
    "preview": (
        str(
            preview_path
        )
        if preview_path is not None
        else None
    ),
    "full_inference_seconds": (
        float(
            full_inference[
                "elapsed_seconds"
            ]
        )
        if full_inference is not None
        else None
    ),
}

manifest_path = (
    OUTPUT_ROOT
    / "pantiny_tiff_inference_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        run_manifest,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(
    "\nManifest saved:",
    manifest_path
)

for name, output_path in full_output_paths.items():
    with rasterio.open(
        output_path
    ) as source:
        print(
            "\n",
            name,
            "→",
            output_path,
        )

        print(
            "Shape:",
            (
                source.count,
                source.height,
                source.width,
            ),
        )

        print(
            "Dtype:",
            source.dtypes,
        )

        print(
            "CRS:",
            source.crs,
        )

        print(
            "Bounds:",
            source.bounds,
        )

        print(
            "Transform:",
            source.transform,
        )

if (
    DELETE_SCRATCH_AFTER_SUCCESS
    and full_inference is not None
):
    prediction_sum = full_inference[
        "prediction_sum"
    ]

    weight_sum = full_inference[
        "weight_sum"
    ]

    del prediction_sum
    del weight_sum
    del full_inference

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    shutil.rmtree(
        SCRATCH_DIR,
        ignore_errors=True,
    )

    print(
        "\nScratch memmap files deleted."
    )

print(
    "\nDONE."
)

print(
    "Results folder:",
    OUTPUT_ROOT
)



## الملفات النهائية

```text
PanTiny_TIFF_Inference_Results/
│
├── Wald_Test_GeoTIFFs/
│   ├── test_.../
│   │   ├── input_lr_ms.tif
│   │   ├── input_pan.tif
│   │   ├── target_hr_ms.tif
│   │   ├── output_pantiny_hr_ms.tif
│   │   └── baseline_bicubic_hr_ms.tif
│   └── wald_test_tiff_manifest.csv
│
├── Full_Scene/
│   ├── PanTiny_HRMS_6band_full_float32.tif
│   ├── PanTiny_HRMS_6band_full_uint16.tif
│   └── PanTiny_HRMS_6band_full_RGB_preview.png
│
└── pantiny_tiff_inference_manifest.json
```

### عند حدوث CUDA Out of Memory

غيّر:

```python
TILE_LR = 128
OVERLAP_LR = 24
STRIDE_LR = TILE_LR - OVERLAP_LR
```

ثم أعد التشغيل من خلية تحديد الـTiles.
